In [1]:
import json
import re
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data/cicids")
STIX_FILE = Path("../data/attck/enterprise-attack.json")
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "cicids_sample_processed.csv"

RANDOM_SEED = 42
SAMPLE_PER_TACTIC = 2000
BENIGN_SAMPLE_SIZE = 2000

In [2]:
# Load ATT&CK STIX data and extract techniques and their associated tactics
with open(STIX_FILE, "r", encoding="utf-8") as f:
    bundle = json.load(f)

techniques = {}
for obj in bundle.get("objects", []):
    if obj.get("type") != "attack-pattern" or obj.get("revoked", False):
        continue
    tech_id = None
    for ref in obj.get("external_references", []):
        if ref.get("source_name") == "mitre-attack":
            tech_id = ref.get("external_id")
            break
    if not tech_id:
        continue
    tactics = [p["phase_name"].replace("-", " ").title() for p in obj.get("kill_chain_phases", []) if p.get("kill_chain_name") == "mitre-attack"]
    techniques[tech_id] = {
        "name": obj.get("name", "Unknown"),
        "tactic": tactics[0] if tactics else "Unknown"
    }
print(f"Loaded {len(techniques)} ATT&CK techniques")

Loaded 703 ATT&CK techniques


In [3]:
# CELL 3: Label Mapping & Text Builder
raw_map = {
    "FTP-Patator": "T1110.001", "FTP Patator": "T1110.001",
    "SSH-Patator": "T1110.001", "SSH Patator": "T1110.001",
    "DoS slowloris": "T1499.001", "DoS Slowhttptest": "T1499.001",
    "DoS Hulk": "T1499.001", "DoS GoldenEye": "T1499.001",
    "Heartbleed": "T1190", "Web Attack Brute Force": "T1110.001",
    "Web Attack XSS": "T1059.007", "Web Attack Sql Injection": "T1190",
    "Infiltration": "T1105", "Bot": "T1071.001",
    "PortScan": "T1046", "DDoS": "T1498.001", "BENIGN": None,
}

label_map = {}
for label, tech_id in raw_map.items():
    if tech_id is None:
        label_map[label] = {"technique_id": "BENIGN", "technique_name": "Benign", "tactic": "Benign"}
    else:
        info = techniques.get(tech_id, {"name": "Unknown", "tactic": "Unknown"})
        label_map[label] = {"technique_id": tech_id, "technique_name": info["name"], "tactic": info["tactic"]}

def normalise_label(label):
    if not isinstance(label, str): return "BENIGN"
    return re.sub(r"\s+", " ", re.sub(r"[^\x00-\x7F]+", " ", label)).strip()

def build_alert_text(row):
    dst_port = int(float(row.get("Destination Port", 0)))
    duration = int(float(row.get("Flow Duration", 0)))
    fwd_pkts = int(float(row.get("Total Fwd Packets", 0)))
    bwd_pkts = int(float(row.get("Total Backward Packets", 0)))
    flow_bps = round(float(row.get("Flow Bytes/s", 0.0)), 2)
    syn_flag = int(float(row.get("SYN Flag Count", 0)))
    ack_flag = int(float(row.get("ACK Flag Count", 0)))
    
    port_map = {21:"FTP", 22:"SSH", 53:"DNS", 80:"HTTP", 443:"HTTPS", 8080:"HTTP-alt"}
    service = port_map.get(dst_port, f"unknown_port_{dst_port}")
    
    behavior = []
    if duration < 100: behavior.append("very_short_connection")
    elif duration > 1_000_000: behavior.append("long_duration")
    if fwd_pkts + bwd_pkts <= 2: behavior.append("low_packet_count")
    elif fwd_pkts + bwd_pkts > 1000: behavior.append("high_packet_count")
    if flow_bps > 1_000_000: behavior.append("high_volume_traffic")
    
    flags = ", ".join([f for f, c in [("SYN", syn_flag), ("ACK", ack_flag)] if c > 0]) or "none"
    return f"Flow to {service}. Behavior: {', '.join(behavior) or 'normal'}. Duration {duration}us. Fwd {fwd_pkts}, Bwd {bwd_pkts}. Bytes/s {flow_bps}. Flags {flags}."

In [4]:
# Load, Map & Build Alert Text
all_rows = []
# Exclude Monday raw file since it contains benign-only
csv_files = sorted([f for f in DATA_DIR.glob("*.csv") if "Monday" not in f.name])

for fp in csv_files:
    df = pd.read_csv(fp, low_memory=False, encoding="latin-1")
    df.columns = df.columns.str.strip()
    df["Label"] = df["Label"].apply(normalise_label)
    
    # Map ATT&CK metadata
    df["attck_technique_id"] = df["Label"].map(lambda x: label_map.get(x, {}).get("technique_id", "UNMAPPED"))
    df["attck_tactic"] = df["Label"].map(lambda x: label_map.get(x, {}).get("tactic", "Unknown"))
    df["alert_text"] = df.apply(build_alert_text, axis=1)
    all_rows.append(df)

combined = pd.concat(all_rows, ignore_index=True)
print(f"Loaded {len(combined)} total rows. Now sampling")

# stratified sampling
attacks = combined[combined["attck_tactic"] != "Benign"].copy()
benign = combined[combined["attck_tactic"] == "Benign"].copy()


sampled_attacks = []
for tactic in sorted(attacks["attck_tactic"].unique()):
    subset = attacks[attacks["attck_tactic"] == tactic]
    n = min(len(subset), SAMPLE_PER_TACTIC)
    sampled_attacks.append(subset.sample(n=n, random_state=RANDOM_SEED))
sampled_attacks_df = pd.concat(sampled_attacks)

sampled_benign_df = benign.sample(n=min(len(benign), BENIGN_SAMPLE_SIZE), random_state=RANDOM_SEED)

final_df = pd.concat([sampled_attacks_df, sampled_benign_df], ignore_index=False)
final_df = final_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
final_df.index.name = "sample_id"
final_df["sample_id"] = final_df.index

# Keep only necessary columns
keep_cols = ["Destination Port", "Flow Duration", "Total Fwd Packets", "Total Backward Packets", 
             "Flow Bytes/s", "SYN Flag Count", "ACK Flag Count", "Label", 
             "attck_technique_id", "attck_tactic", "alert_text", "sample_id"]
final_df = final_df[[c for c in keep_cols if c in final_df.columns]]

Loaded 2300825 total rows. Now sampling


In [8]:
print(final_df["alert_text"].head(20))

sample_id
0     Flow to HTTP. Behavior: long_duration. Duratio...
1     Flow to HTTP-alt. Behavior: high_volume_traffi...
2     Flow to HTTP-alt. Behavior: long_duration. Dur...
3     Flow to FTP. Behavior: very_short_connection, ...
4     Flow to unknown_port_51762. Behavior: very_sho...
5     Flow to SSH. Behavior: very_short_connection, ...
6     Flow to HTTP-alt. Behavior: normal. Duration 6...
7     Flow to HTTP. Behavior: long_duration. Duratio...
8     Flow to HTTP-alt. Behavior: long_duration. Dur...
9     Flow to HTTP. Behavior: long_duration. Duratio...
10    Flow to unknown_port_1. Behavior: very_short_c...
11    Flow to HTTP-alt. Behavior: normal. Duration 9...
12    Flow to HTTP. Behavior: long_duration. Duratio...
13    Flow to unknown_port_2629. Behavior: very_shor...
14    Flow to DNS. Behavior: high_volume_traffic. Du...
15    Flow to unknown_port_10629. Behavior: very_sho...
16    Flow to FTP. Behavior: long_duration. Duration...
17    Flow to unknown_port_554. Behavi

In [5]:
# Validation & Save
print("SAMPLE STATISTICS")
print(f"Total rows: {len(final_df)}")
print("Tactic distribution:")
print(final_df["attck_tactic"].value_counts().to_string())
print("\nSaving final dataset...")

final_df.to_csv(OUTPUT_FILE)
print(f"Saved to {OUTPUT_FILE}")


SAMPLE STATISTICS
Total rows: 10684
Tactic distribution:
attck_tactic
Command And Control    2000
Credential Access      2000
Impact                 2000
Discovery              2000
Benign                 2000
Execution               652
Initial Access           32

Saving final dataset...
Saved to ../data/processed/cicids_sample_processed.csv


In [6]:
# Embeddings
from sentence_transformers import SentenceTransformer

sample = pd.read_csv("../data/embeddings/cicids_sample.csv", index_col="sample_id")
print(sample.shape)  # (12673, 12)

model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

texts = sample["alert_text"].tolist()  # List of 12,673 strings

embeddings = model.encode(
    texts,
    batch_size=64,          
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

assert len(embeddings) == len(sample)
print(f"Shape: {embeddings.shape}")  # (12673, 384)
print(f"Norm check: {np.linalg.norm(embeddings[0]):.4f}")  # 1.0000

np.save("../data/embeddings/cicids_embeddings.npy", embeddings)
json.dump({
    "model": "all-MiniLM-L6-v2",
    "device": "cuda",
    "rows": len(sample),
    "dim": 384,
    "seed": 42,
    "batch_size": 64,
    "normalized": True
}, open("../data/embeddings/cicids_embedding_meta.json", "w"), indent=2)

(10684, 16)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/167 [00:00<?, ?it/s]

Shape: (10684, 384)
Norm check: 1.0000
